# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # x has shape (B, C, H, W) = (batch_size, 1, 28, 28) for MNIST
    B, C, H, W = x.shape
    H_patches = H // patch_size
    W_patches = W // patch_size
    # e.g. patch_size = 7 -> H_patches = W_patches = 4 i.e. we have 4x4 = 16 patches of 7x7 pixels each.
    # H_patches tells us how many patches we have along the height dimension, NOT how the dimension of a single patch.

    # output shape: (B, num_patches, patch_dim) where num_patches = H_patches * W_patches and patch_dim = C * patch_size * patch_size
    # e.g. (B, 16, 49) for patch_size = 7
    # basically (B, T, D) where we have a sequence of T tokens (patches) of dimension D (49 pixels flattened) per batch.
    
    # add new dimensions by artificially splitting the height and width dimensions into patches
    # e.g. we get shape (B, C, 4, 7, 4, 7) for patch_size = 7 instead of (B, C, 28, 28)
    x = x.reshape(B, C, H_patches, patch_size, W_patches, patch_size) # (B, C, H_patches, patch_size, W_patches, patch_size)

    # permute to get (B, H_patches, W_patches, C, patch_size, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5).contiguous()

    # flatten C, patch_size, patch_size into a single dimension to get (B, num_patches, patch_dim)
    x = x.reshape(B, H_patches * W_patches, C * patch_size * patch_size)

    return x

In [3]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        self.projection = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.projection(x_patches)

class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        # (each token is a patch)
        self.positional_embedding = nn.Parameter(torch.zeros(1, num_tokens, d_model))
        # initialize close to 0
        nn.init.trunc_normal_(self.positional_embedding, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.positional_embedding

In [4]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GLUFeedForward(nn.Module):
    """
    GLU-family FFN
    Refer to Equation 6 in Shazeer et al. 2020.

    Supported variants: `GLU`, `Bilinear`, `ReGLU`, `GEGLU`, `SwiGLU`
    """
    supported_variants = ["GLU", "Bilinear", "ReGLU", "GEGLU", "SwiGLU"]

    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        # see Shazeer et al. 2020, Equation 6 for notation
        assert variant in self.supported_variants, f"Unsupported variant {variant}. Supported: {self.supported_variants}"
        self.variant = variant
        self.projW = nn.Linear(d_model, d_ff_gated, bias=False)
        self.projV = nn.Linear(d_model, d_ff_gated, bias=False)
        self.projW2 = nn.Linear(d_ff_gated, d_model, bias=False)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout) # maybe could use a single

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        W_out = self.projW(x)
        V_out = self.projV(x)

        # * is element-wise :)
        if self.variant == "GLU":
            gated_out = F.sigmoid(W_out) * V_out
        elif self.variant == "Bilinear":
            gated_out = W_out * V_out
        elif self.variant == "ReGLU":
            gated_out = F.relu(W_out) * V_out
        elif self.variant == "GEGLU":
            gated_out = F.gelu(W_out) * V_out
        elif self.variant == "SwiGLU":
            gated_out = F.silu(W_out) * V_out

        return self.dropout2(self.projW2(self.dropout1(gated_out)))



In [5]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        # TODO: implement. For attention use nn.MultiHeadAttention 
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout, batch_first=True)
        self.mlp = mlp

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x should have shape (B, T, d_model) = (B, num_patches, d_model)
        x = self.layernorm1(x)
        attn_out, _ = self.attention(x, x, x) # x is query, key and value
        x = x + self.dropout1(attn_out)
        x = self.layernorm2(x)
        mlp_out = self.mlp(x)
        x = x + self.dropout2(mlp_out)
        return x


In [6]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    supported_mlps = ["Baseline"]
    supported_mlps.extend(GLUFeedForward.supported_variants)

    def spawn_mlp(self, mlp_kind: str, d_model: int, d_ff: int, dropout: float) -> nn.Module:
        """
        Returns a MLP module of the given kind with the given hyperparameters.
        We need this because otherwise we would "share" the single MLP across transformer blocks :(
        """
        assert mlp_kind in self.supported_mlps, f"Unsupported mlp kind {mlp_kind}. Supported: {self.supported_mlps}"
        if mlp_kind == "Baseline":
            mlp = FeedForward(d_model, d_ff, dropout)
        else:
            mlp = GLUFeedForward(d_model, int(d_ff * 2/3), dropout, mlp_kind)
        return mlp

    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        self.embedder = nn.Sequential(
            PatchEmbed(patch_dim, d_model),
            PositionalEmbedding(self.num_tokens, d_model)
        )

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=self.spawn_mlp(mlp_kind, d_model, d_ff, dropout),
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        self.output_head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = patchify(x, self.patch_size)
        x = self.embedder(x)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=1) # mean pooling over patches (tokens)
        logits = self.output_head(x)
        return logits

In [7]:
# from https://pytorch.org/docs/stable/notes/mps.html
def get_best_device() -> str:
    if not torch.backends.mps.is_available():
        if not torch.backends.mps.is_built():
            print("MPS not available because the current PyTorch install was not "
                "built with MPS enabled.")
        else:
            print("MPS not available because the current MacOS version is not 12.3+ "
                "and/or you do not have an MPS-enabled device on this machine.")
        device_str = "cuda:0" if torch.cuda.is_available() else "cpu"
    else:
        device_str = "mps"
    return device_str

@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = get_best_device()

In [8]:
from pyexpat import model


def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    # from https://discuss.pytorch.org/t/how-do-i-check-the-number-of-parameters-of-a-model/4325/9
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "mlp_kind": mlp_kind,
        "train_losses": train_losses,
        "test_accs": test_accs,
        "trainable_parameters": trainable_parameters
    }

In [9]:
# NOTE: I omitted device="cpu" in TrainConfig since i wrote get_best_device()
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01)

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = TinyViT.supported_mlps
results = []

for kind in runs:
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    # TODO: print anything you might want here
    print(f"\nRun: {kind} | " )
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)
    print(out)


Run: Baseline | 
[Baseline] epoch 1/5 | test acc: 0.8257
[Baseline] epoch 2/5 | test acc: 0.9185
[Baseline] epoch 3/5 | test acc: 0.9279
[Baseline] epoch 4/5 | test acc: 0.9432
[Baseline] epoch 5/5 | test acc: 0.9509
{'mlp_kind': 'Baseline', 'train_losses': [2.3397796154022217, 2.3794500827789307, 2.411214828491211, 2.2875471115112305, 2.3624415397644043, 2.321101665496826, 2.321974515914917, 2.333644390106201, 2.2992727756500244, 2.28536057472229, 2.2889034748077393, 2.290937900543213, 2.284883975982666, 2.3253471851348877, 2.2897086143493652, 2.295118570327759, 2.315054416656494, 2.2897329330444336, 2.25544810295105, 2.2663073539733887, 2.2718095779418945, 2.286224842071533, 2.272860527038574, 2.3110299110412598, 2.2681519985198975, 2.27608585357666, 2.287936210632324, 2.2491416931152344, 2.273691415786743, 2.2337088584899902, 2.238447666168213, 2.220205783843994, 2.2338762283325195, 2.2319579124450684, 2.242244243621826, 2.227388381958008, 2.2304956912994385, 2.263584613800049, 2.2